In [ ]:
import warnings
import os
import ray
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from arch import arch_model
from functions import estimate_parameters, round_p_value, hplot, plot_parameters_all
from tqdm.auto import tqdm
from dataclasses import asdict

for message in [ 
    'divide by zero encountered in log',
    'invalid value encountered in log',
    'Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default).',
    'Tip',
    'invalid value encountered in scalar multiply',
    'Inequality constraints incompatible',
    'overflow encountered in scalar power',
    'overflow encountered in scalar power',
    'The optimizer returned code 9',
]:
    warnings.filterwarnings( 'ignore', message = message )

In [ ]:
RECOMPUTE = False

In [ ]:
ray.init(
    log_to_driver = False,  # Suppress output (otherwise, the notebook becomes too large)
)

In [ ]:
data = pd.read_parquet( "cache/stock_returns.parquet" )
data = data.pivot( index = 'date', columns = 'qid', values = 'total_return' )

In [ ]:
batch_size = 200
for i in np.arange( 0, data.shape[1], batch_size ):
    ids = data.columns[i:i+batch_size]
    

In [ ]:
filename = "cache/parameters_stocks.parquet"

@ray.remote
def estimate_parameters_remote( y ):
    return estimate_parameters( y )

if os.path.exists(filename) and not RECOMPUTE:
    parameters = pd.read_parquet(filename)
else:
    parameters = {}

    batch_size = 200
    for i in tqdm( np.arange( 0, data.shape[1], batch_size ) ):
        ids = data.columns[i:i+batch_size]
        tmp = {}
        for id in ids:
            y = data[id].dropna()
            y = np.log( y ).diff().dropna()
            y = y[ np.isfinite(y) ]
            tmp[id] = estimate_parameters_remote.remote( y )
        tmp = { k: ray.get(v) for k, v in tmp.items() }
        tmp = { k: asdict(v) for k, v in tmp.items() }
        parameters = parameters | tmp

    parameters = pd.DataFrame( parameters ).T
    parameters.to_parquet(filename)

In [ ]:
plot_parameters_all( parameters )

print( 
    f"Only {int(100 * np.mean( parameters['denominator'] > 0 ))}% "
    "of time series satisfy the condition 1-α²κ-2αβ-β²>0" 
)